In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
experiment_folder = Path("experiment_output")
folders = {
    "BattleZone": experiment_folder / "Rainbow_BattleZone200M",
    "NameThisGame": experiment_folder / "Rainbow_NameThisGame200M",
    "Phoenix": experiment_folder / "Rainbow_Phoenix200M",
}

# Data Inspection

In [ ]:
for game in ["BattleZone", "NameThisGame", "Phoenix"]:
    eval_data = np.load(
    folders[game] / f"Rainbow_{game}200M_Evaluation.npy",
    allow_pickle=False,
    )
    print(game, eval_data.shape)
    print("Last non-zero row:", np.flatnonzero(np.any(eval_data != 0, axis=1))[-1])
    print("Last three rows:")
    print(eval_data[-3:])

In [ ]:
row_nonzero = np.any(eval_data != 0, axis=1)

print("Rows containing any non-zero score:")
print(np.flatnonzero(row_nonzero))

print("Last row containing a non-zero score:")
print(np.flatnonzero(row_nonzero)[-1])

In [ ]:
completed_evals = eval_data[:250]

In [ ]:
# eval_data must already be loaded.
# Rows 0 to 249 are completed evaluations.
# Rows 250 and 251 are unused.
completed_evals = eval_data[:250]

# 250 evaluations distributed across 200 million Atari frames.
evaluation_frames = (
    np.arange(1, len(completed_evals) + 1) * 800_000
)

# Statistics for each evaluation.
eval_mean = completed_evals.mean(axis=1)
eval_median = np.median(completed_evals, axis=1)
eval_std = completed_evals.std(axis=1, ddof=1)
eval_min = completed_evals.min(axis=1)
eval_max = completed_evals.max(axis=1)

print(
    "Completed evaluations: {}".format(
        len(completed_evals)
    )
)

print(
    "Episodes per evaluation: {}".format(
        completed_evals.shape[1]
    )
)

print(
    "Final evaluation frame: {:,.0f}".format(
        evaluation_frames[-1]
    )
)

print("\nFinal evaluation:")

print(
    "Mean:   {:,.1f}".format(
        eval_mean[-1]
    )
)

print(
    "Median: {:,.1f}".format(
        eval_median[-1]
    )
)

print(
    "SD:     {:,.1f}".format(
        eval_std[-1]
    )
)

print(
    "Range:  {:,.0f} to {:,.0f}".format(
        eval_min[-1],
        eval_max[-1]
    )
)

# Find the evaluation with the highest mean score.
best_index = int(np.argmax(eval_mean))

print("\nBest evaluation:")

print(
    "Index:  {}".format(
        best_index
    )
)

print(
    "Frames: {:.1f}M".format(
        evaluation_frames[best_index] / 1_000_000
    )
)

print(
    "Mean:   {:,.1f}".format(
        eval_mean[best_index]
    )
)

print(
    "Median: {:,.1f}".format(
        eval_median[best_index]
    )
)

print(
    "SD:     {:,.1f}".format(
        eval_std[best_index]
    )
)

print(
    "Range:  {:,.0f} to {:,.0f}".format(
        eval_min[best_index],
        eval_max[best_index]
    )
)

# Combine the final ten evaluations to provide a more stable
# summary of performance near the end of training.
final_n = 10
final_window = completed_evals[-final_n:]
final_scores = final_window.reshape(-1)

print(
    "\nFinal {} evaluations combined:".format(
        final_n
    )
)

print(
    "Frame range: {:.1f}M to {:.1f}M".format(
        evaluation_frames[-final_n] / 1_000_000,
        evaluation_frames[-1] / 1_000_000
    )
)

print(
    "Episodes: {}".format(
        len(final_scores)
    )
)

print(
    "Mean:    {:,.1f}".format(
        final_scores.mean()
    )
)

print(
    "Median:  {:,.1f}".format(
        np.median(final_scores)
    )
)

print(
    "SD:      {:,.1f}".format(
        final_scores.std(ddof=1)
    )
)

print(
    "Range:   {:,.0f} to {:,.0f}".format(
        final_scores.min(),
        final_scores.max()
    )
)

# Results

In [ ]:
games = [
    "BattleZone",
    "NameThisGame",
    "Phoenix",
]

completed_evaluation_count = 250
episodes_per_evaluation = 20
total_atari_frames = 200_000_000
evaluation_interval = (
    total_atari_frames // completed_evaluation_count
)

summaries = []
curves = {}


for game in games:
    evaluation_file = (
        folders[game]
        / f"Rainbow_{game}200M_Evaluation.npy"
    )

    experiment_file = (
        folders[game]
        / f"Rainbow_{game}200M_Experiment.npy"
    )

    evaluation_data = np.load(
        evaluation_file,
        allow_pickle=False,
    )

    experiment_data = np.load(
        experiment_file,
        allow_pickle=False,
    )

    if evaluation_data.shape != (252, 20):
        raise ValueError(
            "{} has unexpected evaluation shape {}".format(
                game,
                evaluation_data.shape,
            )
        )

    if np.any(evaluation_data[250:] != 0):
        raise ValueError(
            "{} has unexpected data after row 249".format(
                game
            )
        )

    completed_evaluations = evaluation_data[:250]

    evaluation_frames = (
        np.arange(
            1,
            completed_evaluation_count + 1,
        )
        * evaluation_interval
    )

    evaluation_mean = completed_evaluations.mean(axis=1)

    evaluation_median = np.median(
        completed_evaluations,
        axis=1,
    )

    evaluation_std = completed_evaluations.std(
        axis=1,
        ddof=1,
    )

    evaluation_min = completed_evaluations.min(axis=1)
    evaluation_max = completed_evaluations.max(axis=1)

    standard_error = (
        evaluation_std
        / np.sqrt(episodes_per_evaluation)
    )

    confidence_interval_95 = 1.96 * standard_error

    best_index = int(np.argmax(evaluation_mean))

    final_window = completed_evaluations[-10:]
    final_window_scores = final_window.reshape(-1)

    final_agent_steps = experiment_data[-1, 1]
    final_recorded_atari_frames = final_agent_steps * 4

    summaries.append(
        {
            "Game": game,
            "Final mean": evaluation_mean[-1],
            "Final median": evaluation_median[-1],
            "Final SD": evaluation_std[-1],
            "Final minimum": evaluation_min[-1],
            "Final maximum": evaluation_max[-1],
            "Best mean": evaluation_mean[best_index],
            "Best median": evaluation_median[best_index],
            "Best SD": evaluation_std[best_index],
            "Best frame": evaluation_frames[best_index],
            "Final-10 mean": final_window_scores.mean(),
            "Final-10 median": np.median(final_window_scores),
            "Final-10 SD": final_window_scores.std(ddof=1),
            "Final-10 minimum": final_window_scores.min(),
            "Final-10 maximum": final_window_scores.max(),
            "Recorded Atari frames": final_recorded_atari_frames,
        }
    )

    curves[game] = {
        "frames": evaluation_frames,
        "mean": evaluation_mean,
        "median": evaluation_median,
        "ci95": confidence_interval_95,
    }


summary_table = pd.DataFrame(summaries)

summary_table.to_csv(
    "rainbow_validation_summary.csv",
    index=False,
)


display_columns = [
    "Game",
    "Final mean",
    "Final median",
    "Final SD",
    "Best mean",
    "Best frame",
    "Final-10 mean",
]

formatters = {
    "Final mean": "{:,.1f}".format,
    "Final median": "{:,.1f}".format,
    "Final SD": "{:,.1f}".format,
    "Best mean": "{:,.1f}".format,
    "Best frame": "{:,.0f}".format,
    "Final-10 mean": "{:,.1f}".format,
}

print(
    summary_table[display_columns].to_string(
        index=False,
        formatters=formatters,
    )
)

In [ ]:
figure, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(15, 4.5),
)

for axis, game in zip(axes, folders):
    curve = curves[game]

    frames_in_millions = (
        curve["frames"] / 1_000_000
    )

    axis.plot(
        frames_in_millions,
        curve["mean"],
        linewidth=1.5,
        label="Mean evaluation return",
    )

    axis.fill_between(
        frames_in_millions,
        curve["mean"] - curve["ci95"],
        curve["mean"] + curve["ci95"],
        alpha=0.2,
        label="Approximate 95% CI",
    )

    axis.set_title(game)
    axis.set_xlabel("Atari frames, millions")
    axis.set_ylabel("Evaluation return")
    axis.set_xlim(0, 200)
    axis.grid(alpha=0.2)

handles, labels = axes[0].get_legend_handles_labels()

figure.suptitle(
    "Rainbow evaluation performance over 200 million Atari frames",
    y=1.02,
)

figure.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.97),
    ncol=2,
    frameon=False,
)

figure.tight_layout(
    rect=(0, 0, 1, 0.88)
)

figure.savefig(
    "rainbow_validation_learning_curves.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()